In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

load_dotenv()

True

In [2]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
parser = StrOutputParser()

In [ ]:
key_facts_prompt = PromptTemplate(
    template="List 5 key facts about the following topic:\n\n{topic}",
    input_variables=["topic"],
)

counterargs_prompt = PromptTemplate(
    template="List 5 counterarguments related to the following topic:\n\n{topic}",
    input_variables=["topic"],
)

key_facts_chain = key_facts_prompt | llm | parser
counterargs_chain = counterargs_prompt | llm | parser

In [4]:
parallel_chain = RunnableParallel(
    key_facts=key_facts_chain,
    counterarguments=counterargs_chain,
)

In [ ]:
def merge_outputs(parallel_result):
    merged = ( "Facts:\n" + parallel_result["key_facts"] + "\n\nCounterarguments:\n" + parallel_result["counterarguments"] )
    return merged

merge_step = RunnableLambda(merge_outputs)

In [6]:
report_prompt = PromptTemplate(
    template=(
        "Using the research notes below, write a balanced and structured research report.\n"
        "Include an introduction, a section on key facts, a section on counterarguments, and a conclusion.\n\n"
        "Research Notes:\n{merged}"
    ),
    input_variables=["merged"],
)

report_chain = report_prompt | llm | parser

In [ ]:
full_chain = ( RunnablePassthrough() | parallel_chain | merge_step | (lambda merged: {"merged": merged}) | report_chain )

In [8]:
topic = "Artificial Intelligence in Healthcare"

print(f"Generating report for: {topic}\n")
report = full_chain.invoke({"topic": topic})
print(report)

Generating report for: Artificial Intelligence in Healthcare

**Introduction**

The integration of Artificial Intelligence (AI) in healthcare has the potential to revolutionize the way medical services are delivered, making them more efficient, effective, and personalized. AI can analyze vast amounts of medical data, provide accurate diagnoses, and tailor treatment plans to individual patients. However, like any emerging technology, AI in healthcare also raises several concerns and criticisms. This report aims to provide a balanced view of the benefits and challenges of AI in healthcare, highlighting key facts, counterarguments, and potential implications for the future of healthcare.

**Key Facts**

The use of AI in healthcare has several key benefits, including:

1. **Improved Diagnosis**: AI can analyze large amounts of medical data to help doctors diagnose diseases more accurately and quickly.
2. **Personalized Medicine**: AI can help tailor treatment plans to individual patients b

In [ ]:
filename = topic.lower().replace(" ", "_") + "_report.txt"

with open(filename, "w") as f:
    f.write(f"Research Report: {topic}\n")
    f.write("=" * 60 + "\n\n")
    f.write(report)

print(f"Report saved to {filename}")

Report saved to artificial_intelligence_in_healthcare_report.txt
